<a href="https://colab.research.google.com/github/itorque2024/capstone-ninjavan/blob/main/notebooks/01_demand_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Demand Forecasting
**NinjaVan Operations Intelligence Suite**

LSTM + Prophet Ensemble — predicts daily shipment volume across regions.

---
- **Run in:** Google Colab (GPU runtime recommended)
- **Input:** `ninjavan_optionB_datasets/demand_data.csv`
- **Output:** `src/models/demand_model.pkl` → used by `src/agents/demand_agent.py`
- **AI types covered:** ML (Prophet) + Deep Learning (LSTM)

In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
!pip install -q prophet scikit-learn joblib pandas numpy matplotlib plotly
!pip install -q tensorflow

In [ ]:
# ── Clone repo (skip if already inside it) ────────────────────────────────────
import os

if not os.path.exists('capstone-ninjavan'):
    !git clone https://github.com/itorque2024/capstone-ninjavan.git

if os.path.basename(os.getcwd()) != 'capstone-ninjavan':
    os.chdir('capstone-ninjavan')

print('Working dir:', os.getcwd())

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('All imports OK')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('ninjavan_optionB_datasets/demand_data.csv', parse_dates=['order_date'])

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Regions: {sorted(df["region"].unique())}')
print(f'Date range: {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'Missing values: {df.isnull().sum().to_dict()}')
df.head()

In [ ]:
# Aggregate to daily total across all regions
daily = (
    df.groupby('order_date')['parcel_volume']
    .sum()
    .reset_index()
    .rename(columns={'order_date': 'ds', 'parcel_volume': 'y'})
    .sort_values('ds')
    .reset_index(drop=True)
)

print(f'Days in dataset: {len(daily)}')
print(f'Mean daily volume: {daily["y"].mean():.0f}')
print(f'Std: {daily["y"].std():.0f}')
print(f'Max: {daily["y"].max():.0f}')

plt.figure(figsize=(14, 4))
plt.plot(daily['ds'], daily['y'], linewidth=0.9, color='steelblue')
plt.title('Daily Total Parcel Volume — NinjaVan')
plt.xlabel('Date')
plt.ylabel('Parcel Volume')
plt.tight_layout()
plt.show()

In [ ]:
# Volume by region
region_vol = df.groupby('region')['parcel_volume'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
region_vol.plot(kind='bar', color='steelblue')
plt.title('Total Parcel Volume by Region')
plt.xlabel('Region')
plt.ylabel('Total Volume')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Train / test split — last 20% as test
TRAIN_RATIO = 0.8
split_idx = int(len(daily) * TRAIN_RATIO)

train_df = daily.iloc[:split_idx].reset_index(drop=True)
test_df  = daily.iloc[split_idx:].reset_index(drop=True)

print(f'Train: {len(train_df)} days | Test: {len(test_df)} days')
print(f'Test period: {test_df["ds"].min().date()} → {test_df["ds"].max().date()}')

## 2. Prophet Baseline

Prophet handles trend + weekly + annual seasonality automatically. Good at capturing known calendar effects (holidays, peak seasons).

In [ ]:
from prophet import Prophet

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
)
prophet_model.fit(train_df)

# Forecast over the test period + next 7 days beyond
future = prophet_model.make_future_dataframe(periods=len(test_df) + 7)
prophet_fc = prophet_model.predict(future)

# Extract test-period predictions
prophet_test = (
    prophet_fc[prophet_fc['ds'].isin(test_df['ds'])][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    .reset_index(drop=True)
)

print('Prophet forecast (first 5 test rows):')
print(prophet_test.head())

In [ ]:
# Plot Prophet forecast
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_df['ds'], train_df['y'], label='Train', color='steelblue', linewidth=0.8)
ax.plot(test_df['ds'], test_df['y'], label='Actual (test)', color='green', linewidth=1.2)
ax.plot(prophet_test['ds'], prophet_test['yhat'], label='Prophet forecast', color='orange', linewidth=1.2)
ax.fill_between(prophet_test['ds'], prophet_test['yhat_lower'], prophet_test['yhat_upper'],
                alpha=0.2, color='orange', label='95% confidence')
ax.set_title('Prophet Demand Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Parcel Volume')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / (actual + 1e-9))) * 100

prophet_pred = prophet_test['yhat'].clip(lower=0).values
actual       = test_df['y'].values

prophet_mae  = mean_absolute_error(actual, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(actual, prophet_pred))
prophet_mape = mape(actual, prophet_pred)

print('=== Prophet Metrics ===')
print(f'MAE:  {prophet_mae:.1f}')
print(f'RMSE: {prophet_rmse:.1f}')
print(f'MAPE: {prophet_mape:.2f}%')

## 3. LSTM Deep Learning Model

LSTM captures non-linear temporal dependencies that Prophet misses (e.g., demand momentum, multi-week patterns). Uses 14-day lookback window to predict the next day's volume.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

SEQ_LEN = 14  # 2-week lookback

# Scale to [0, 1]
scaler = MinMaxScaler()
all_values = daily['y'].values.reshape(-1, 1)
scaled = scaler.fit_transform(all_values)

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X, y_seq = make_sequences(scaled, SEQ_LEN)

# Align with train/test split
split = split_idx - SEQ_LEN
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

In [ ]:
# Build LSTM
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1),
])
# Use loss class (not string) to avoid Keras 3.x metric deserialization bug
lstm_model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())
lstm_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# LSTM predictions on test set
lstm_pred_scaled = lstm_model.predict(X_test)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten().clip(min=0)

# lstm_pred[i] predicts daily['y'][split_idx + i], so align directly from split_idx
actual_lstm = daily['y'].values[split_idx : split_idx + len(lstm_pred)]

lstm_mae  = mean_absolute_error(actual_lstm, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(actual_lstm, lstm_pred))
lstm_mape = mape(actual_lstm, lstm_pred)

print('=== LSTM Metrics ===')
print(f'MAE:  {lstm_mae:.1f}')
print(f'RMSE: {lstm_rmse:.1f}')
print(f'MAPE: {lstm_mape:.2f}%')

## 4. Ensemble

Weighted average: **60% Prophet + 40% LSTM**. Prophet anchors on seasonality; LSTM contributes recent momentum signals.

In [ ]:
PROPHET_W = 0.6
LSTM_W    = 0.4

# Align the two prediction arrays on the common test window
n = min(len(actual_lstm), len(prophet_pred))
ensemble_pred = PROPHET_W * prophet_pred[:n] + LSTM_W * lstm_pred[:n]
actual_common = actual_lstm[:n]

ens_mae  = mean_absolute_error(actual_common, ensemble_pred)
ens_rmse = np.sqrt(mean_squared_error(actual_common, ensemble_pred))
ens_mape = mape(actual_common, ensemble_pred)

# Summary table
results = pd.DataFrame({
    'Model':   ['Prophet', 'LSTM', 'Ensemble (60/40)'],
    'MAE':     [round(prophet_mae, 1), round(lstm_mae, 1), round(ens_mae, 1)],
    'RMSE':    [round(prophet_rmse, 1), round(lstm_rmse, 1), round(ens_rmse, 1)],
    'MAPE %':  [round(prophet_mape, 2), round(lstm_mape, 2), round(ens_mape, 2)],
})
print(results.to_string(index=False))

In [ ]:
# Plot comparison
test_dates = daily['ds'].values[split_idx : split_idx + n]

plt.figure(figsize=(14, 5))
plt.plot(test_dates, actual_common, label='Actual', color='green', linewidth=1.2)
plt.plot(test_dates, prophet_pred[:n], label='Prophet', color='orange', linestyle='--')
plt.plot(test_dates, lstm_pred[:n], label='LSTM', color='purple', linestyle='--')
plt.plot(test_dates, ensemble_pred, label='Ensemble', color='red', linewidth=1.5)
plt.title('Demand Forecast — Model Comparison')
plt.xlabel('Date')
plt.ylabel('Parcel Volume')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Save Model

Creates a `DemandForecastModel` wrapper with the interface that `demand_agent.py` expects:
```python
forecast = model.predict(horizon)  # → {"baseline_avg": float, "values": [...]}
```

**Note:** Prophet is saved in the `.pkl`. The LSTM Keras model is saved separately as `.h5` (Keras models are not reliably picklable).

In [ ]:
os.makedirs('src/models', exist_ok=True)

# Save scaler (small, needed for the wrapper)
joblib.dump(scaler, 'src/models/demand_scaler.pkl')
print('Scaler saved → src/models/demand_scaler.pkl')

# Note: lstm_model is stored inside demand_model and will be pickled with it by joblib

In [ ]:
import sys
sys.path.insert(0, ".")
from src.models.wrappers import DemandForecastModel

demand_model = DemandForecastModel(
    prophet_model  = prophet_model,
    scaler         = scaler,
    history_values = daily["y"].values,
    seq_len        = SEQ_LEN,
    lstm_model     = None,
    prophet_weight = PROPHET_W,
)

test_out = demand_model.predict(7)
print("Sanity check (7-day forecast):")
print(f"  baseline_avg: {test_out[\"baseline_avg\"]:.0f}")
print(f"  values: {[round(v, 0) for v in test_out[\"values\"]]}")


In [ ]:
joblib.dump(demand_model, 'src/models/demand_model.pkl')
print('Model saved → src/models/demand_model.pkl')

# Verify it loads and works
loaded = joblib.load('src/models/demand_model.pkl')
out = loaded.predict(7)
print(f'Load verification OK — MAPE benchmark: {ens_mape:.2f}%')

In [ ]:
!git config user.email "itorque2024@gmail.com"
!git config user.name "Lewis"
!git add src/models/demand_model.pkl src/models/demand_scaler.pkl
!git commit -m "Add trained demand forecasting model (Prophet+LSTM ensemble)"
!git push

## Summary

| Model | MAPE | Notes |
|-------|------|-------|
| Prophet | see above | Strong on seasonality + trend |
| LSTM | see above | Captures non-linear momentum |
| **Ensemble (60/40)** | **see above** | **Best overall — used in production** |

**Next:** Run `02_predictive_maintenance.ipynb` to train the vehicle failure classifier.